# 01 — Pipeline ETI end-to-end (step-by-step)

**Track:** `docs/experiment/RESEARCH_TRACK.md` · **Oleada 1**

Ejemplifica Extract → Transform → Infer → retrieval con la API pública.
Seed / dogfood — **no** PRODUCT §5.

Requisitos: Neo4j, `ungraph[infer-en]`, modelo `en_core_web_sm`.

In [ ]:
from pathlib import Path
import ungraph
from ungraph.core.configuration import reset_configuration

REPO = Path("../../..").resolve()
CORPUS = REPO / "benchmarks/domains/knowledge_graphs/corpus/kg_survey.md"
assert CORPUS.is_file(), CORPUS

reset_configuration()
ungraph.configure(inference_mode="ner")  # + neo4j_* desde .env
print("ungraph", ungraph.__version__, "| mode", ungraph.get_settings().inference_mode)

## Extract + Transform + persist (+ Infer según mode)

In [ ]:
info = ungraph.infer_over_document(CORPUS, chunk_size=800, chunk_overlap=120)
info

## Post-ingest: mine + topology + stats

In [ ]:
print(ungraph.mine_knowledge())
print(ungraph.validate_topology())
stats = ungraph.graph_stats()
stats

## Retrieval (interfaz, no definición de conocimiento)

In [ ]:
hits = ungraph.hybrid_search("knowledge graph entity linking", limit=5)
for h in hits:
    print(round(h.score, 3), (h.content or "")[:120].replace("\n", " "))

## Figura mínima — conteos estructurales

Si `stats` expone labels/rels, graficar; si no, imprimir keys.

In [ ]:
import matplotlib.pyplot as plt

labels = stats.get("labels") or stats.get("node_counts") or {}
if isinstance(labels, dict) and labels:
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(list(labels.keys()), list(labels.values()))
    ax.set_title("Node counts by label")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    out = REPO / "benchmarks/domains/knowledge_graphs/reports/research"
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / "01_node_counts.png", dpi=120)
    plt.show()
else:
    print("stats keys:", list(stats.keys()) if isinstance(stats, dict) else type(stats))